## Melody Features

This notebook demonstrates the basic usage of _melody-features_ in two ways:

1. **Individual features** — load a `Melody` and call a feature function (for example `pitch_range`).
2. **Batch extraction** — run `get_all_features` to compute the full feature set for every melody in a directory or file list.

The sections below walk through loading melodies first, then batch usage and configuration. 

## Loading melodies

Melodies are represented as `Melody` objects. Load one from a MIDI path, several from a directory, or use the bundled Essen corpus:

In [6]:
from melody_features import load_melodies_from_directory, essen_corpus
from melody_features.corpus import get_corpus_files
from melody_features.io.midi import load_midi

# Single file from the bundled Essen corpus
midi_path = get_corpus_files("essen", max_files=1)[0]
melody = load_midi(str(midi_path))

# Or load many at once from a folder of .mid files
# melodies = load_melodies_from_directory(str(essen_corpus), file_type="midi")

print(f"Melody ID: {melody.id},\nNumber of notes: {len(melody.pitches)}")

Melody ID: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/appenzel.mid,
Number of notes: 140


## Computing individual features

Feature functions are exported from the top-level `melody_features` package. Pass the arguments named in each function's docstring (here, the note pitch list):

In [9]:
from melody_features import pitch_range

val = pitch_range(melody.pitches)

print(f"Pitch range: {val}")

Pitch range: 24


## Batch extraction with `get_all_features`

`get_all_features` computes a wide range of features on every melody in the input, returning a single DataFrame with one row per melody.

In [10]:
from melody_features import get_all_features
from melody_features.corpus import get_corpus_files

first_ten_essen = get_corpus_files('essen', max_files=10)

# IDyOM can take a long time, so we have the option to skip it if we like
features = get_all_features(first_ten_essen, skip_idyom=True)
features.head()


13:38:31 - melody_features - INFO - Starting feature extraction job...
13:38:31 - melody_features - INFO - Configuration Parameters:
13:38:31 - melody_features - INFO -   Key Estimation Strategy: infer_if_necessary
13:38:31 - melody_features - INFO -   Key Finding Algorithm: krumhansl_schmuckler
13:38:31 - melody_features - INFO -   Corpus Path: /Users/davidwhyatt/feature_set/src/melody_features/corpora/pearce_default_idyom
13:38:31 - melody_features - INFO -   IDyOM Configurations: 4 config(s)
13:38:31 - melody_features - INFO -     [pitch_stm]:
13:38:31 - melody_features - INFO -       Models: :stm
13:38:31 - melody_features - INFO -       Corpus: Using Corpus Path from Config
13:38:31 - melody_features - INFO -       Target Viewpoints: ['cpitch']
13:38:31 - melody_features - INFO -       Source Viewpoints: [('cpitch', 'cpint', 'cpintfref')]
13:38:31 - melody_features - INFO -       PPM Order: None
13:38:31 - melody_features - INFO -     [pitch_ltm]:
13:38:31 - melody_features - INFO

,melody_num,melody_id,absolute_pitch.basic_pitch_histogram,absolute_pitch.first_pitch,absolute_pitch.importance_of_bass_register,absolute_pitch.importance_of_high_register,absolute_pitch.importance_of_middle_register,absolute_pitch.interval_between_most_prevalent_pitches,absolute_pitch.last_pitch,absolute_pitch.mean_pitch,...,corpus.tfdf_kendall,corpus.mean_log_tfdf,corpus.norm_log_dist,corpus.max_log_df,corpus.min_log_df,corpus.mean_log_df,corpus.mean_global_local_weight,corpus.std_global_local_weight,corpus.mean_global_weight,corpus.std_global_weight
0,1,/Users/davidwhyatt/feature_set/src/melody_feat...,"{62: 9, 64: 3, 66: 4, 67: 10, 69: 27, 71: 25, ...",62,0.0,0.350000,0.650000,2,79,71.757143,...,0.272013,0.000031,0.004997,9.78136,0.0,4.427284,1.278345,0.538920,0.998849,0.043322
1,2,/Users/davidwhyatt/feature_set/src/melody_feat...,"{66: 3, 67: 9, 69: 8, 70: 4, 72: 3, 74: 14, 75...",74,0.0,0.801471,0.198529,2,67,77.220588,...,0.321697,0.000037,0.004492,9.78136,0.0,4.461493,1.474110,0.711033,0.994205,0.034579
2,3,/Users/davidwhyatt/feature_set/src/melody_feat...,"{55: 1, 57: 4, 58: 9, 60: 17, 62: 31, 64: 37, ...",62,0.0,0.024793,0.975207,2,62,65.161157,...,0.387192,0.000009,0.002554,9.78136,0.0,3.942758,1.289015,0.675330,0.992263,0.035933
3,4,/Users/davidwhyatt/feature_set/src/melody_feat...,"{57: 1, 61: 1, 62: 5, 64: 8, 66: 11, 67: 6, 68...",57,0.0,0.000000,1.000000,3,62,66.384615,...,0.422580,0.000079,0.009250,9.78136,0.0,4.563073,1.125190,0.398173,0.990498,0.011373
4,5,/Users/davidwhyatt/feature_set/src/melody_feat...,"{55: 1, 57: 5, 58: 5, 59: 1, 60: 6, 62: 5, 63: 1}",58,0.0,0.000000,1.000000,3,60,59.250000,...,0.363190,0.000362,0.019982,9.78136,0.0,4.963442,1.156905,0.363188,0.991678,0.018893


We can save the results of the feature calculations to a csv file using the below:

In [ ]:
features.to_csv('output.csv')

The feature set has a few customisable aspects that change the behaviour of some of the feature calculations. There is no requirement to customise this configuration, as sensible values recommended in the literature are supplied as defaults. However, for users seeking more control over the behaviour of FANTASTIC and IDyOM, the pipeline `Config` dataclass is provided in `melody_features.pipeline.config` (with `IDyOMConfig` in `melody_features.idyom.config`):

In [16]:
# Import the Config dataclasses
from melody_features.pipeline.config import Config, FantasticConfig
from melody_features.idyom.config import IDyOMConfig

print(f"Config: {Config}")
# Print out all parameters of the Config dataclass
for field, field_obj in Config.__dataclass_fields__.items():
    value = getattr(Config, field_obj.name, None)
    print(f"{field}: {value}")

Config: <class 'melody_features.pipeline.config.Config'>
idyom: None
fantastic: None
corpus: None
key_estimation: infer_if_necessary
key_finding_algorithm: krumhansl_schmuckler


Once we import these dataclasses, we can begin to customise our configuration:

In [17]:
# Initialise the config object with the relevant parameters
from melody_features import essen_corpus

config = Config(
    corpus=essen_corpus,
    idyom={"pitch": IDyOMConfig(
        target_viewpoints=["cpitch"],
        source_viewpoints=[("cpint", "cpintfref")],
        ppm_order=1,
        models=":both"
    )},
        fantastic=FantasticConfig(
        max_ngram_order=5,
        phrase_gap=1.5
    ),
    key_estimation="always_read_from_file",
    key_finding_algorithm="krumhansl_schmuckler"
)

The `corpus` parameter operates on different levels. If you wish to use the same corpus for both FANTASTIC and IDyOM, you need only set it in the top level of `Config()`; you then do not need to supply it to `IDyOMConfig` or `FantasticConfig`. 

If you want to use different corpora for each different toolbox, `IDyOMConfig` and `FantasticConfig` will override whatever is supplied in the top level of `Config`.

In [ ]:
# Initialise the config object with different corpora
different_corpus_config = Config(
    corpus=None, # will be overridden
    idyom={"pitch": IDyOMConfig(
        target_viewpoints=["cpitch"],
        source_viewpoints=[("cpint", "cpintfref")],
        ppm_order=None,
        models=":both",
        corpus=None
    )},
        fantastic=FantasticConfig(
        max_ngram_order=2,
        phrase_gap=1.5,
        corpus=essen_corpus
    ),
    key_estimation="always_read_from_file",
    key_finding_algorithm="krumhansl_schmuckler"
)

features_different_corpus = get_all_features(first_ten_essen, config=different_corpus_config)
features_different_corpus.head()

13:40:48 - melody_features - INFO - Starting feature extraction job...
13:40:48 - melody_features - INFO - Configuration Parameters:
13:40:48 - melody_features - INFO -   Key Estimation Strategy: always_read_from_file
13:40:48 - melody_features - INFO -   Key Finding Algorithm: krumhansl_schmuckler
13:40:48 - melody_features - INFO -   Corpus Path: None (corpus features disabled)
13:40:48 - melody_features - INFO -   IDyOM Configurations: 1 config(s)
13:40:48 - melody_features - INFO -     [pitch]:
13:40:48 - melody_features - INFO -       Models: :both
13:40:48 - melody_features - INFO -       Corpus: Using Corpus Path from Config
13:40:48 - melody_features - INFO -       Target Viewpoints: ['cpitch']
13:40:48 - melody_features - INFO -       Source Viewpoints: [('cpint', 'cpintfref')]
13:40:48 - melody_features - INFO -       PPM Order: None
13:40:48 - melody_features - INFO -   FANTASTIC Configuration:
13:40:48 - melody_features - INFO -     Max N-gram Order: 2
13:40:48 - melody_fea

** Putting Test dataset files in experiment history folder. **
** Putting Pretraining dataset files in experiment history folder. **
** running lisp script **
To load "clsql":
  Load 1 ASDF system:
    clsql
; Loading "clsql"

To load "idyom":
  Load 1 ASDF system:
    idyom
; Loading "idyom"
................

Inserting 10 compositions into database: dataset 66082526134054.
| Progress: -----------------------------------------------|
Inserting 903 compositions into database: dataset 99082526134054.
| Progress: -----------------------------------------------|

We can also supply multiple IDyOM configurations, allowing us to compute information content using different 'viewpoints' or corpora in one run of the feature set. This can be achieved like so:

In [ ]:
multi_idyom_config = Config(
    corpus=essen_corpus,
    idyom={"pitch": IDyOMConfig(
        target_viewpoints=["cpitch"],
        source_viewpoints=[("cpint", "cpintfref")],
        ppm_order=None,
        models=":both",
        corpus=essen_corpus
    ),
    "rhythm": IDyOMConfig(
        target_viewpoints=["onset"],
        source_viewpoints=["ioi"],
        ppm_order=None,
        models=":both",
        corpus=None
    )},
        fantastic=FantasticConfig(
        max_ngram_order=5,
        phrase_gap=1.5,
        corpus=None
    ),
    key_estimation="always_read_from_file",
    key_finding_algorithm="krumhansl_schmuckler"
)

In [ ]:
# Now we can get the different IDyOM features along with everything else
# (this will take a while)
features_different_idyom = get_all_features(first_ten_essen, config=multi_idyom_config)
features_different_idyom.head()

As well as skipping corpus-dependent features, we can choose to skip IDyOM entirely if we like, as it can be quite time-consuming if you don't intend to use its output:

In [ ]:
features_no_idyom = get_all_features(first_ten_essen, skip_idyom=True)
features_no_idyom.head()

There are two other methods for `key_estimation`: `infer_if_necessary` and `always_infer`. These settings rely on `key_finding_algorithm` to estimate the key of the melody. Specifically, `infer_if_necessary` will attempt to read the key from the MIDI file, and if it is unable to detect key information, will estimate it using the method from `key_finding_algorithm`. `always_infer` will ignore any key information in the MIDI file and estimate it in the same fashion. 